In [1]:
from decouple import config
from sqlalchemy import create_engine
import pandas as pd
from datetime import datetime, timedelta, time, date
import time 
from sqlalchemy import text
import oracledb
import sys

In [2]:
#CONEXIONES
DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="essi_etl"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena1  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine1 = create_engine(cadena1)
connection1 = engine1.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dw_essalud"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena2  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine2 = create_engine(cadena2)
connection2 = engine2.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="etl_logs"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [3]:
fecha_ini = pd.read_sql_query("SELECT fec_ini FROM etl_act where id_mod=24", con=connection2)
fecha_ini= fecha_ini.iloc[0, 0]

fecha_fin = pd.read_sql_query("SELECT fec_ter FROM etl_act where id_mod=24", con=connection2)
fecha_fin= fecha_fin.iloc[0, 0]


# Definir el número total de meses desde la fecha de inicio hasta la fecha actual
total_meses = (fecha_fin.year - fecha_ini.year) * 12 + (fecha_fin.month - fecha_ini.month)

In [4]:
for i in range(total_meses+1):
	inicioTiempo = time.time()
	now_inicio = datetime.now()

	# Calcular la fecha de inicio y fin del intervalo mensual
	fecha_ini_mes = fecha_ini
		
	# Obtener el último día del mes actual
	if fecha_fin.month==fecha_ini.month and fecha_fin.year==fecha_ini.year:
		fecha_fin_mes = fecha_fin
	else :
		ultimo_dia_mes_actual = date(fecha_ini_mes.year, fecha_ini_mes.month, 1) + timedelta(days=32)
		fecha_fin_mes = ultimo_dia_mes_actual.replace(day=1)

	fecha_ini_str = fecha_ini_mes.strftime('%Y-%m-%d')
	fecha_fin_str = fecha_fin_mes.strftime('%Y-%m-%d')

	print(f"Procesando 2.1 de {fecha_ini_str} al {fecha_fin_str}")


	now = datetime.now()

	query=f"UPDATE etl_act SET fec_act ='{now}' WHERE id_mod=24"

	c1= text(query)
	connection2.execute(c1)
	
	tabla='dat_teledot001_essi'

	oracledb.init_oracle_client()
	oracledb.version = "8.3.0"
	sys.modules["cx_Oracle"] = oracledb
	un = config("USER4_BDI_POSTGRES")
	pw = config("PASS4_BDI_POSTGRES")
	hostname=config("HOST4_BDI_POSTGRES")
	service_name="WNET"
	port = 1527

	engine0 = create_engine(f'oracle://{un}:{pw}@',connect_args={
			"host": hostname,
			"port": port,
			"service_name": service_name
		}
	)

	connection0 = engine0.connect()

	query = f"""SELECT
	a.atenprocenasicod                               AS CODCENTRO, 
	ca.cenasides                                     AS CENTRO,
	to_char(z.atenproproperfec,'yyyymm')             AS PERIODO,
	b.servhoscod                                     AS COD_SERVICIO,
	b.servhosdes                                     AS SERVICIO,
	d.actcod                                         AS COD_ACTIVIDAD,
	D.ACTDES                                         AS ACTIVIDAD,
	e.actespcod                                      AS COD_SUBACTIVIDAD,
	E.ACTESPNOM                                      AS SUBACTIVIDAD,
	a.atenproperasisdocidennum                       AS DNI_MEDICO,
	p.perasisproapepat || ' ' ||
	p.perasisproapemat|| ' ' || p.perasisproprinom   AS PROFESIONAL,
	s.perautcod                                      AS AUTOGENERADO,
	s.perdocidennum                                  AS DNI,
	s.perapepatdes || ' ' || s.perapematdes
	|| ' ' || s.perprinomdes                         AS PACIENTE,
	(FLOOR(MONTHS_BETWEEN(z.atenproproperfec,s.pernacfec)/12))                    AS ANNOS,
	(FLOOR(MOD(MONTHS_BETWEEN(z.atenproproperfec,s.pernacfec),12)))               AS MESES,
	(TO_DATE(z.atenproproperfec, 'DD/MM/YYYY')- ADD_MONTHS(s.pernacfec,
	FLOOR(MONTHS_BETWEEN(TO_DATE(z.atenproproperfec,'DD/MM/YYYY'),s.pernacfec)))) AS DIAS,
	decode(s.persexocod,'1','M','0','F','')                                       AS SEXO,
	s.percaldomnom || ' ' || s.pernmkdomnum || ' ' || s.perinldomnum
	|| ' ' ||s.perurbdomnom                                                      AS DIRECCION,
	c.pachisclinum                                                                AS H_C,
	m.tipsegdes                                                                   AS TIPO_SEGURO,
	(select tp.tipoparenom from cbtpa10 tp
	WHERE tp.tipoparecod = s.pertipoparecod)                                      AS TIPO_PARENTESCO, 
	n.tipopacinom                                                                 AS TIPO_PACIENTE,
	to_char(t.citambsolfec, 'YYYY-MM-DD')                                         AS FECHA_SOLIC,
	to_char(z.atenproproperfec, 'YYYY-MM-DD')                                     AS FECHA_ATEN,
	decode(t.condcitacod,'1','NORMAL','2','ADICIONAL')                            AS CONDICION_CITA,
	z.atenproactmednum                                                            AS ACTO_MED,
	a.atenprcpscod                                                                AS CODPROCED,
	replace(replace(trim(to_char(substr(f.cpsdes ,0,4000))),CHR(10),''),CHR(13),'')    AS PROCEDIMIENTO,
	a.atenprdcant                                                                      AS CANTPROCED,
	z1.cenasioricod                                                                    AS COD_PRECEDENCIA,
	s.percenasiadscod                                                                  AS CAS_ADSCRIPCION,
	ct.concod                                                                          AS COD_CONSULTORIO,
	(decode(pr.tipohorprogcod,'1','NORMAL','2','EXTRAS','3','RPCT',
	'4','EXTRAS+CITAS','5','COMPENSACION'))                                            AS TIP_PROGRAMACION, 
	to_char(pr.properturhorini, 'hh24:mi')                                             AS horaini,
	to_char(pr.properturhorfin, 'hh24:mi')                                             AS horafin,
	to_char(pr.properturhorini, 'hh24:mi')||'-'||to_char(pr.properturhorfin,'hh24:mi') AS TURNO,
	decode(k.actmedestgrav,'0','NO GESTANTE','1','GESTANTE')                           AS TIPO_GRAVIDEZ,
	s.perrucempnum                                                                     AS NUM_RUC,
	a.atenprdresprcovid                                                                AS CODRESUL_COVID,
	(SELECT rc.resprcoviddes FROM sgss.cbrpr10 rc
	WHERE rc.resprcovidcod = a.atenprdresprcovid)                                      AS RESULT_COVID,
	c.pactelfij                                                                        AS TELEF_FIJO, 
	c.pactelcel                                                                        AS TELEF_MOVIL, 
	decode(a.atenprdtipenf,'0','PACIENTE NO COVID','1','PACIENTE COVID')               AS ESTADO_PACIENTE,
	a.atenprdusucrecod                                                                 AS USU_REG,  
	to_char(a.atenprdcrefec,'YYYY-MM-DD')                                              AS FECH_REG,
	to_char(a.atenprdcrefec,'hh24:mi')                                                 AS HORA_REG,
	a.atenprdusumodcod                                                                 AS USU_MODIF,
	to_char(a.atenprdmodfec,'YYYY-MM-DD')                                              AS FECH_MODIF,
	to_char(a.atenprdmodfec,'hh24:mi')                                                 AS HORA_MODIF,
	a.atenprdtipteledot                                                                AS TIPTELEDOT,
	replace(replace(trim( to_char(substr(A.ATENPRDOBS,0,4000))),
	CHR(10), ''), CHR(13), '')                                                         AS OBSERVACRESULTADO
	from ctapd10 a
	LEFT OUTER JOIN CMCAS10 ca ON ca.oricenasicod = a.atenprooricenasicod
							AND ca.cenasicod   = a.atenprocenasicod
	left outer join ctapr10 z on z.atenprooricenasicod = a.atenprooricenasicod
								and z.atenprocenasicod   = a.atenprocenasicod
								and z.atenproarehoscod   = a.atenproarehoscod
								and z.atenproservhoscod  = a.atenproservhoscod
								and z.atenproactcod      = a.atenproactcod
								and z.atenproactespcod   = a.atenproactespcod
								and z.atenprotipdocidenpercod   = a.atenprotipdocidenpercod
								and z.atenproperasisdocidennum  = a.atenproperasisdocidennum
								and z.atenproproperfec          = a.atenproproperfec
								and z.atenproturinihor          = a.atenproturinihor
								and z.atenproturfinhor          = a.atenproturfinhor
								and z.atenproactmednum          = a.atenproactmednum
	left outer join cmame10 k on z.atenprooricenasicod = k.oricenasicod
								and z.atenprocenasicod   = k.cenasicod
								and z.atenproactmednum   = k.actmednum
	left outer join cmtse10 m on k.actmedtipsegcod     = m.tipsegcod
	left outer join cmper10 s on k.actmedpacsecnum     = s.persecnum
	left outer join ctcam10 t on z.atenprooricenasicod = t.citamboricenasicod
								and z.atenprocenasicod   = t.citambcenasicod
								and z.atenproactmednum   = t.citambnum
	left outer join cmcpp10 f on  a.atenprcpscod              = f.cpscod
	left outer join cmsho10 b on a.atenproservhoscod          = b.servhoscod
	LEFT OUTER JOIN CMACT10 D ON A.ATENPROACTCOD = D.ACTCOD
	LEFT OUTER JOIN CMACE10 E ON A.ATENPROACTCOD = E.ACTCOD
								AND A.ATENPROACTESPCOD = E.ACTESPCOD
	left outer join cbtpc10 n on k.actmedtipopacicod          = n.tipopacicod
	left outer join cmprs10 p on a.atenprotipdocidenpercod    = p.tipdocidenpercod
								and a.atenproperasisdocidennum  = p.perasisdocidennum
	left outer join cmpac10 c on c.oricenasicod               = k.oricenasicod
								and c.cenasicod                 = k.cenasicod
								and c.pacsecnum                 = k.actmedpacsecnum
	LEFT OUTER JOIN ctref10 z1 ON k.actmedorirefnum                = z1.refnum 
	left outer join ctpco10 ct on ct.proconoricenasicod = t.citambproconoricenasicod
							and ct.proconcenasicod   = t.citambcenasicod
							and ct.proconarehoscod   = t.citambarehoscod
							and ct.proconservhoscod  = t.citambservhoscod
							and ct.proconactcod      = t.citambactcod
							and ct.proconactespcod   = t.citambactespcod
							and ct.procontipdocidenpercod = t.citambtipdocidenpercod
							and ct.proconperasisdocidennum = t.citambperasisdocidennum
							and ct.proconfec               = t.citambproconfec
							and ct.proconturhorini         = t.citambproconturhorini
							and ct.proconturhorfin         = t.citambproconturhorfin
	left outer join ctppe10 pr on pr.oricenasicod       = ct.proconoricenasicod
							and pr.cenasicod          = ct.proconcenasicod
							and pr.arehoscod          = ct.proconarehoscod
							and pr.servhoscod         = ct.proconservhoscod
							and pr.actcod             = ct.proconactcod
							and pr.actespcod          = ct.proconactespcod
							and pr.tipdocidenpercod   = ct.procontipdocidenpercod
							and pr.perasisdocidennum  = ct.proconperasisdocidennum
							and pr.properfec          = ct.proconfec
							and pr.properturhorini    = ct.proconturhorini
							and pr.properturhorfin    = ct.proconturhorfin    
	where
		a.atenproarehoscod                      = '01' 
	and to_char(trunc(A.ATENPROPROPERFEC), 'YYYY-MM_DD') >= '{fecha_ini_str}'
	and to_char(trunc(A.ATENPROPROPERFEC), 'YYYY-MM_DD') < '{fecha_fin_str}'
	and a.atenproservhoscod IN ('F11', 'F19')
	order by  b.servhosdes,          d.actdes,          e.actespnom,          p.perasisproapepat || ' ' || p.perasisproapemat || ' ' || p.perasisproprinom,          f.cpsdes
	"""
	base1 = pd.read_sql_query(query, con=connection0)


	connection0.close()


	borrando=f"DELETE FROM {tabla} WHERE FECHA_ATEN >='{fecha_ini_str}' and FECHA_ATEN <'{fecha_fin_str}'"
	borrado = connection2.execute(borrando)

	base1.to_sql(name=f'{tabla}', con=engine2, if_exists='append', index=False)

	query=f"UPDATE etl_act SET fec_ini ='{now}' WHERE id_mod=24"

	c1= text(query)
	connection2.execute(c1)

	fecha_ini = fecha_fin_mes

	finproceso=time.time()
	tiempoproceso=finproceso - inicioTiempo
	tiempoproceso=round(tiempoproceso,3)
	print("Proceso completado satisfactoriamente en " + str(tiempoproceso)+" segundos")

Procesando 2.1 de 2024-07-09 al 2024-08-01
Proceso completado satisfactoriamente en 1319.095 segundos
Procesando 2.1 de 2024-08-01 al 2024-09-01
Proceso completado satisfactoriamente en 299.486 segundos
Procesando 2.1 de 2024-09-01 al 2024-10-01
Proceso completado satisfactoriamente en 296.078 segundos
Procesando 2.1 de 2024-10-01 al 2024-11-01
Proceso completado satisfactoriamente en 316.342 segundos
Procesando 2.1 de 2024-11-01 al 2024-12-01
Proceso completado satisfactoriamente en 269.437 segundos
Procesando 2.1 de 2024-12-01 al 2024-12-31
Proceso completado satisfactoriamente en 257.358 segundos
